# 拡張演習 — 自分でMCPツールを追加する
### モジュール1〜3のまとめとして、AWSワークショップの前に

これまでのモジュールでは、多くの部分がすでに用意されていました。この演習では、
**設計から実装まで、できるだけ自分の力で**やってみましょう。

**今日のシナリオ:** ニンバス・ロボティクスの在庫担当者から、こんな要望が来ました——
「在庫が少なくなっている商品を、ひと目で確認したい。」

現在のMCPサーバーには `check_inventory`(1つの商品IDを指定して確認)しかありません。
**在庫が少ない商品を一覧で返す、新しいツールを追加してください。**

**この演習が終わったら:**
- 自分の判断でMCPツールを設計・実装する経験ができます
- 新しいツールを追加しただけでは、エージェントがそれを使ってくれるとは
  限らないことを確認します(判断ロジックも一緒に更新する必要があります)
- 午後のAWSワークショップ(AgentCore)で、同じ「ツールを追加する」という
  作業が、マネージドサービスの上でどう変わるかを比較する土台になります


## 環境セットアップ(SageMaker ノートブックインスタンス)

- **インスタンスタイプ:** `ml.t3.medium` で十分です
- **カーネル:** `conda_pytorch_p310`

In [1]:
# このノートブックインスタンスで一度だけ実行してください
%pip install --quiet faiss-cpu scikit-learn 'fastmcp>=4.0'

Note: you may need to restart the kernel to use updated packages.


## セットアップ:モジュール1〜3を持ち込む

これはモジュール3の最後の状態(RAG検索・MCPサーバー・エージェントループ)を
そのまま再現したものです。完成済みのコードとして提供します。

In [2]:
import asyncio, json, re
from fastmcp import FastMCP, Client
import numpy as np
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

def TODO(hint=""):
    """演習の未完成部分を示す関数です。TODO(...)の呼び出しを自分のコードに
    置き換えてください。置き換えるまではエラーが出続けます(これは正常な動作です)。"""
    raise NotImplementedError(f"ここを実装してください。ヒント: {hint}")

# --- モジュール1から: RAG ---
DOCS = [
    "ニンバス・スカウトドローンの飛行時間は20分で、価格は249ドルです。",
    "ニンバス・カーゴドローンは最大5キログラムまで運搬でき、価格は899ドルです。",
    "返品は購入から30日以内で、元の梱包があれば受け付けます。",
    "ニンバス・ロボティクスの保証は、製造上の欠陥について12か月間保証します。",
    "ニンバスのドローンのバッテリーをフル充電するには約90分かかります。",
]

def chunk_documents(docs, max_chars=60):
    chunks = []
    for doc in docs:
        for i in range(0, len(doc), max_chars):
            chunks.append(doc[i:i+max_chars])
    return chunks

def embed_chunks_local(chunks):
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
    vectors = vectorizer.fit_transform(chunks).toarray().astype("float32")
    return vectors, vectorizer

def retrieve(query, vectorizer, index, chunks, k=2):
    qvec = vectorizer.transform([query]).toarray().astype("float32")
    distances, indices = index.search(qvec, k)
    return [chunks[i] for i in indices[0]]

chunks = chunk_documents(DOCS)
vectors, vectorizer = embed_chunks_local(chunks)
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

# --- モジュール2から: MCPサーバー ---
INVENTORY = {
    "drone-100": {"name": "ニンバス・スカウトドローン", "stock": 12, "price": 249.00},
    "drone-200": {"name": "ニンバス・カーゴドローン", "stock": 3, "price": 899.00},
}

mcp = FastMCP("NimbusInventory")

@mcp.tool
def check_inventory(product_id: str) -> dict:
    """ニンバス・ロボティクスの製品IDを指定して、在庫数と価格を確認する。"""
    item = INVENTORY.get(product_id)
    if not item:
        return {"error": f"Unknown product_id '{product_id}'"}
    return item

# --- モジュール3から: 判断・生成・エージェントループ ---
def mock_llm_decide(user_query: str) -> str:
    product_match = re.search(r"drone-\d+", user_query)
    if "在庫" in user_query and product_match:
        return json.dumps({"tool_call": "check_inventory", "args": {"product_id": product_match.group()}})
    if any(word in user_query for word in ["保証", "返品", "充電", "飛行時間"]):
        return json.dumps({"retrieve": user_query})
    return json.dumps({"response": "製品に関する質問、在庫状況、ポリシーについてお答えできます。"})

def mock_llm_generate(user_query: str, context_chunks: list) -> str:
    context = " ".join(context_chunks)
    return f"次の情報が見つかりました: {context}"

async def agent_loop(user_query: str) -> str:
    raw = llm_decide(user_query)
    decision = json.loads(raw)
    if "tool_call" in decision:
        async with Client(mcp) as client:
            result = await client.call_tool(decision["tool_call"], decision["args"])
            return f"ツールの結果: {result.data}"
    elif "retrieve" in decision:
        found = retrieve(decision["retrieve"], vectorizer, index, chunks)
        return mock_llm_generate(user_query, found)
    else:
        return decision["response"]

print("モジュール1〜3の部品を読み込みました。ツール一覧:", ["check_inventory"])

モジュール1〜3の部品を読み込みました。ツール一覧: ['check_inventory']


## タスク1: 新しいツールを設計する

**要件:**
- ツール名: `list_low_stock`
- 引数: `threshold`(在庫数のしきい値、整数、デフォルト値 `5`)
- 動作: `INVENTORY` の中から、在庫数が `threshold` **未満**の商品だけを、
  リスト形式で返す(各商品は `product_id` を含む辞書)
- 適切なdocstringをつけること(モジュール2で学んだコツを思い出してください:
  動詞から始める・引数の形式を明記する)

まずは、自然に思いつく書き方で実装してみましょう。

In [ ]:
@mcp.tool
def list_low_stock(threshold: int = 5) -> list:
    """在庫数がthreshold未満の製品を一覧で返す。"""
    # TODO: INVENTORYを1件ずつ確認し、stockがthreshold未満の商品を
    # {"product_id": ..., ...商品の情報...} という辞書のリストとして返してください
    TODO("INVENTORY.items()をループし、item['stock'] < threshold の商品だけをリストに集める")


In [3]:
# --- レスキューセル ---
@mcp.tool
def list_low_stock(threshold: int = 5) -> list:
    """在庫数がthreshold未満の製品を一覧で返す。"""
    return [
        {"product_id": pid, **item}
        for pid, item in INVENTORY.items()
        if item["stock"] < threshold
    ]

## テストしてみましょう

`list_low_stock` を呼び出してみます。ドローン一覧(在庫12と3)のうち、
しきい値5未満なのはカーゴドローン(在庫3)だけのはずです。

In [4]:
async def test_tool():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print("利用可能なツール:", [t.name for t in tools])
        result = await client.call_tool("list_low_stock", {"threshold": 5})
        print("result.data:", result.data)

await test_tool()

利用可能なツール: ['check_inventory', 'list_low_stock']
result.data: None


## ⚠️ 何かおかしいと思いませんか?

`result.data` が `None` になっていませんか? ツール自体は動いているはずなのに、
結果が受け取れていません。

**調べてみましょう:** `result.data` の代わりに `result.content` を見てみてください。

In [5]:
async def debug_tool():
    async with Client(mcp) as client:
        result = await client.call_tool("list_low_stock", {"threshold": 5})
        print("result.content:", result.content)

await debug_tool()

result.content: [TextContent(type='text', text='[{"product_id":"drone-200","name":"ニンバス・カーゴドローン","stock":3,"price":899.0}]', annotations=None, meta=None)]


**種明かし:** `result.content` を見ると、正しいデータ(JSON形式のテキスト)が
ちゃんと存在しています。つまり、ツール自体は正しく動いています。問題は
`result.data` に**構造化されたデータとして**変換されていないことです。

**原因は戻り値の型ヒントです。** `-> list` という書き方は、Pythonとしては正しい
のですが、FastMCPに対しては「リストの中身が何なのか」という情報が足りません。
モジュール2で学んだ「型ヒントとdocstringがツールの仕様書になる」という話を、
**引数だけでなく戻り値にも**当てはめる必要がある、という新しい発見です。

**修正方法:** `-> list` ではなく、`-> list[dict]` のように、**リストの中身の型**
まで具体的に書きます。

In [6]:
# --- レスキューセル(修正版) ---
@mcp.tool
def list_low_stock(threshold: int = 5) -> list[dict]:
    """在庫数がthreshold未満の製品を一覧で返す。"""
    return [
        {"product_id": pid, **item}
        for pid, item in INVENTORY.items()
        if item["stock"] < threshold
    ]

async def test_tool_fixed():
    async with Client(mcp) as client:
        result = await client.call_tool("list_low_stock", {"threshold": 5})
        print("result.data:", result.data)

await test_tool_fixed()

[09/14/26 22:01:44] WARNING  Component already exists: tool:list_low_stock@                   ]8;id=11385829;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=11385830;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

result.data: [{'product_id': 'drone-200', 'name': 'ニンバス・カーゴドローン', 'stock': 3, 'price': 899.0}]


## タスク2: エージェントに新しいツールを教える

ツールは正しく動くようになりました。では、モジュール3のエージェントに
質問してみましょう。

In [7]:
llm_decide = mock_llm_decide
print(await agent_loop("在庫が少ない商品はありますか?"))

製品に関する質問、在庫状況、ポリシーについてお答えできます。


**気づきましたか?** ツールは正しく動くのに、エージェントは関係のない回答を
返したはずです。**新しいツールを追加しただけでは、エージェントはそれを
使ってくれません。** `llm_decide` がこの質問をどう判断するかを、まだ
教えていないからです。

**あなたの番:** `mock_llm_decide` に新しい分岐を追加してください。
「在庫が少ない」「少なくなって」のような言葉が含まれていたら、
`{"tool_call": "list_low_stock", "args": {"threshold": 5}}` を返すようにします。

In [ ]:
def mock_llm_decide(user_query: str) -> str:
    product_match = re.search(r"drone-\d+", user_query)
    if "在庫" in user_query and product_match:
        return json.dumps({"tool_call": "check_inventory", "args": {"product_id": product_match.group()}})
    # TODO: ここに新しい分岐を追加してください。
    # 「少ない」または「少なくなって」が user_query に含まれていたら、
    # {"tool_call": "list_low_stock", "args": {"threshold": 5}} を返す
    TODO('if "少ない" in user_query or "少なくなって" in user_query: return json.dumps({...})')
    if any(word in user_query for word in ["保証", "返品", "充電", "飛行時間"]):
        return json.dumps({"retrieve": user_query})
    return json.dumps({"response": "製品に関する質問、在庫状況、ポリシーについてお答えできます。"})

llm_decide = mock_llm_decide
print(await agent_loop("在庫が少ない商品はありますか?"))

In [8]:
# --- レスキューセル ---
def mock_llm_decide(user_query: str) -> str:
    product_match = re.search(r"drone-\d+", user_query)
    if "在庫" in user_query and product_match:
        return json.dumps({"tool_call": "check_inventory", "args": {"product_id": product_match.group()}})
    if "少ない" in user_query or "少なくなって" in user_query:
        return json.dumps({"tool_call": "list_low_stock", "args": {"threshold": 5}})
    if any(word in user_query for word in ["保証", "返品", "充電", "飛行時間"]):
        return json.dumps({"retrieve": user_query})
    return json.dumps({"response": "製品に関する質問、在庫状況、ポリシーについてお答えできます。"})

llm_decide = mock_llm_decide
print(await agent_loop("在庫が少ない商品はありますか?"))
print(await agent_loop("drone-200の在庫はありますか?"))  # 既存のツールも壊れていないか確認
print(await agent_loop("保証期間はどれくらいですか?"))    # RAGの経路も壊れていないか確認

ツールの結果: [{'product_id': 'drone-200', 'name': 'ニンバス・カーゴドローン', 'stock': 3, 'price': 899.0}]
ツールの結果: {'name': 'ニンバス・カーゴドローン', 'stock': 3, 'price': 899.0}
次の情報が見つかりました: ニンバス・ロボティクスの保証は、製造上の欠陥について12か月間保証します。 ニンバス・スカウトドローンの飛行時間は20分で、価格は249ドルです。


## まとめ

このモジュールでは、比較的少ないヒントで、自分の力でMCPツールを1つ設計・実装
しました。その過程で、次の2つの教訓に出会いました:

1. **戻り値にも型ヒントの具体性が必要。** `list` だけでは足りず、`list[dict]`
   のように中身まで書く必要がある(FastMCPが構造化データを正しく返すため)
2. **ツールを追加しただけでは、エージェントは賢くならない。** 判断ロジック
   (`llm_decide`)も一緒に更新して、初めて新しいツールが活きてきます

午後のAWSワークショップでは、この「ツールを追加する」という同じ作業が、
AgentCoreというマネージドサービスの上でどう変わるかを見ていきます。